# 04. Weather Data Integration

## Objective

This notebook performs the preprocessing and validation of the hourly weather dataset obtained from Open-Meteo. The cleaned weather data will later be merged with the Delhi Metro hourly ridership dataset for forecasting analysis.

### Tasks

- Import required libraries
- Load weather dataset
- Inspect dataset structure
- Validate timestamps
- Check missing values
- Check duplicate records
- Verify data types
- Prepare dataset for merging

In [10]:
# ==========================================================
# Import Required Libraries
# ==========================================================

# Data manipulation
import pandas as pd
import numpy as np

# Data visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Date and time handling
from datetime import datetime
from pathlib import Path

# Ignore unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.2f}".format)

# Plot settings
plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 6)

print("Libraries imported successfully.")

Libraries imported successfully.


In [11]:
# ==========================================================
# Define Reproducible Project Paths and Load Metadata
# ==========================================================

project_root = Path.cwd().resolve()
if project_root.name == 'Notebooks':
    project_root = project_root.parent

weather_input_path = project_root / 'DataRaw' / 'Weather_RC_5month.csv'

metadata = pd.read_csv(
    weather_input_path,
    nrows=1
)

metadata

,latitude,longitude,elevation,utc_offset_seconds,timezone,timezone_abbreviation
0,28.65,77.17,214.00,19800,Asia/Kolkata,GMT+5:30


In [12]:
weather = pd.read_csv(
    weather_input_path,
    skiprows=3
)

print("Weather dataset loaded successfully.")

Weather dataset loaded successfully.


## Initial Data Inspection

In this section, we examine the overall structure of the weather dataset before performing any cleaning or preprocessing. The objective is to understand the dataset dimensions, column names, data types, and summary statistics, while identifying any obvious issues that may require further attention.

In [13]:
# ==========================================================
# Dataset Dimensions
# ==========================================================

print(f"Number of rows    : {weather.shape[0]}")
print(f"Number of columns : {weather.shape[1]}")

Number of rows    : 4344
Number of columns : 11


In [14]:
# ==========================================================
# Preview Dataset
# ==========================================================

display(weather.head())

display(weather.tail())

,time,temperature_2m (°C),relative_humidity_2m (%),apparent_temperature (°C),precipitation (mm),rain (mm),weather_code (wmo code),surface_pressure (hPa),cloud_cover (%),visibility (m),wind_speed_10m (km/h)
0,2024-09-01T00:00,27.20,90,33.40,0.00,0.00,0,980.60,4,24140.00,5.10
1,2024-09-01T01:00,27.10,89,33.00,0.00,0.00,0,980.00,2,24140.00,5.90
2,2024-09-01T02:00,26.70,91,32.40,0.00,0.00,0,979.70,2,24140.00,6.90
3,2024-09-01T03:00,26.40,91,32.00,0.00,0.00,0,979.80,1,24140.00,7.20
4,2024-09-01T04:00,26.20,92,31.80,0.00,0.00,0,980.00,1,24140.00,7.20


,time,temperature_2m (°C),relative_humidity_2m (%),apparent_temperature (°C),precipitation (mm),rain (mm),weather_code (wmo code),surface_pressure (hPa),cloud_cover (%),visibility (m),wind_speed_10m (km/h)
4339,2025-02-28T19:00,21.50,82,23.10,0.00,0.00,1,986.20,28,24140.00,9.40
4340,2025-02-28T20:00,20.20,87,22.60,0.90,0.90,53,987.40,28,11680.00,2.50
4341,2025-02-28T21:00,18.30,91,20.30,6.30,6.30,63,988.20,20,11980.00,2.70
4342,2025-02-28T22:00,18.60,90,20.10,0.00,0.00,2,988.00,67,24140.00,6.30
4343,2025-02-28T23:00,18.20,91,19.00,0.00,0.00,2,988.30,56,24140.00,10.60


In [15]:
# ==========================================================
# Column Names
# ==========================================================

print("Weather Dataset Columns:\n")

for i, column in enumerate(weather.columns, start=1):
    print(f"{i}. {column}")

Weather Dataset Columns:

1. time
2. temperature_2m (°C)
3. relative_humidity_2m (%)
4. apparent_temperature (°C)
5. precipitation (mm)
6. rain (mm)
7. weather_code (wmo code)
8. surface_pressure (hPa)
9. cloud_cover (%)
10. visibility (m)
11. wind_speed_10m (km/h)


In [16]:
# ==========================================================
# Dataset Information
# ==========================================================

weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 4344 entries, 0 to 4343
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   time                       4344 non-null   str    
 1   temperature_2m (°C)        4344 non-null   float64
 2   relative_humidity_2m (%)   4344 non-null   int64  
 3   apparent_temperature (°C)  4344 non-null   float64
 4   precipitation (mm)         4344 non-null   float64
 5   rain (mm)                  4344 non-null   float64
 6   weather_code (wmo code)    4344 non-null   int64  
 7   surface_pressure (hPa)     4344 non-null   float64
 8   cloud_cover (%)            4344 non-null   int64  
 9   visibility (m)             4344 non-null   float64
 10  wind_speed_10m (km/h)      4344 non-null   float64
dtypes: float64(7), int64(3), str(1)
memory usage: 441.3 KB


In [17]:
# ==========================================================
# Summary Statistics
# ==========================================================

weather.describe().T

,count,mean,std,min,25%,50%,75%,max
temperature_2m (°C),4344.00,20.87,6.71,6.60,15.30,21.10,26.00,35.20
relative_humidity_2m (%),4344.00,68.20,19.92,19.00,53.75,71.00,85.00,100.00
apparent_temperature (°C),4344.00,21.68,8.49,4.80,14.70,21.10,29.20,40.90
precipitation (mm),4344.00,0.10,0.71,0.00,0.00,0.00,0.00,13.20
rain (mm),4344.00,0.10,0.71,0.00,0.00,0.00,0.00,13.20
weather_code (wmo code),4344.00,4.19,13.77,0.00,0.00,0.00,1.00,65.00
surface_pressure (hPa),4344.00,987.94,4.69,973.70,985.50,988.90,991.20,998.20
cloud_cover (%),4344.00,21.38,36.49,0.00,0.00,0.00,25.00,100.00
visibility (m),4344.00,23436.97,2937.96,300.00,24140.00,24140.00,24140.00,24140.00
wind_speed_10m (km/h),4344.00,6.47,3.50,0.00,4.00,5.90,8.30,24.80


## Column Standardisation

The source column labels include spaces, punctuation, and measurement units. To make subsequent analysis and merging reproducible, a working copy is created with concise, unit-explicit snake_case names. The raw `weather` dataframe remains unchanged.

In [18]:
# ==========================================================
# Create Analysis Copy and Standardise Column Names
# ==========================================================

weather_clean = weather.copy()

column_mapping = {
    'time': 'time',
    'temperature_2m (°C)': 'temperature_2m_c',
    'relative_humidity_2m (%)': 'relative_humidity_2m_pct',
    'apparent_temperature (°C)': 'apparent_temperature_c',
    'precipitation (mm)': 'precipitation_mm',
    'rain (mm)': 'rain_mm',
    'weather_code (wmo code)': 'weather_code_wmo',
    'surface_pressure (hPa)': 'surface_pressure_hpa',
    'cloud_cover (%)': 'cloud_cover_pct',
    'visibility (m)': 'visibility_m',
    'wind_speed_10m (km/h)': 'wind_speed_10m_kmh'
}

missing_source_columns = set(column_mapping) - set(weather_clean.columns)
if missing_source_columns:
    raise KeyError(f'Expected source columns are missing: {missing_source_columns}')

weather_clean = weather_clean.rename(columns=column_mapping)
print('Column names standardised successfully.')
weather_clean.columns.tolist()

Column names standardised successfully.


['time',
 'temperature_2m_c',
 'relative_humidity_2m_pct',
 'apparent_temperature_c',
 'precipitation_mm',
 'rain_mm',
 'weather_code_wmo',
 'surface_pressure_hpa',
 'cloud_cover_pct',
 'visibility_m',
 'wind_speed_10m_kmh']

## Timestamp Conversion and Timezone Validation

Open-Meteo provides the observation times as local clock times and records the timezone in the metadata header. We first parse the timestamp field and explicitly verify that the source timezone is `Asia/Kolkata`. The `time` column is intentionally retained as timezone-naive Indian Standard Time so that it can be matched directly to the metro dataset, provided that its timestamps use the same convention.

In [19]:
# ==========================================================
# Convert Timestamp and Validate Source Timezone
# ==========================================================

weather_clean['time'] = pd.to_datetime(weather_clean['time'], errors='coerce')

invalid_timestamps = weather_clean['time'].isna().sum()
if invalid_timestamps:
    raise ValueError(f'{invalid_timestamps} timestamp(s) could not be parsed.')

source_timezone = metadata.loc[0, 'timezone']
expected_timezone = 'Asia/Kolkata'

if source_timezone != expected_timezone:
    raise ValueError(
        f'Unexpected source timezone: {source_timezone}. Expected: {expected_timezone}.'
    )

print(f'Source timezone validated: {source_timezone}')
print(f'Timestamp range: {weather_clean["time"].min()} to {weather_clean["time"].max()}')

Source timezone validated: Asia/Kolkata
Timestamp range: 2024-09-01 00:00:00 to 2025-02-28 23:00:00


## Data Quality Validation

Before exporting the cleaned dataset, the following checks assess completeness, timestamp uniqueness, data types, date coverage, and hourly continuity. These checks are performed before any imputation or other corrective treatment, so that the original data quality is documented transparently.

In [20]:
# ==========================================================
# Missing-Value Assessment
# ==========================================================

missing_summary = pd.DataFrame({
    'missing_count': weather_clean.isna().sum(),
    'missing_percentage': (weather_clean.isna().mean() * 100).round(2)
})

display(missing_summary[missing_summary['missing_count'] > 0])
print(f'Total missing values: {missing_summary["missing_count"].sum()}')

,missing_count,missing_percentage


Total missing values: 0


In [21]:
# ==========================================================
# Duplicate Timestamp Assessment
# ==========================================================

duplicate_timestamp_count = weather_clean['time'].duplicated().sum()
print(f'Duplicate timestamps: {duplicate_timestamp_count}')

if duplicate_timestamp_count:
    display(weather_clean.loc[weather_clean['time'].duplicated(keep=False)].sort_values('time'))
    raise ValueError('Duplicate timestamps must be resolved before merging.')

Duplicate timestamps: 0


In [22]:
# ==========================================================
# Data-Type Verification
# ==========================================================

expected_numeric_columns = [column for column in weather_clean.columns if column != 'time']
non_numeric_columns = [
    column for column in expected_numeric_columns
    if not pd.api.types.is_numeric_dtype(weather_clean[column])
]

if non_numeric_columns:
    raise TypeError(f'Expected numeric weather columns: {non_numeric_columns}')

print('All weather measurement columns have numeric data types.')
weather_clean.dtypes.to_frame(name='data_type')

All weather measurement columns have numeric data types.


,data_type
time,datetime64[us]
temperature_2m_c,float64
relative_humidity_2m_pct,int64
apparent_temperature_c,float64
precipitation_mm,float64
rain_mm,float64
weather_code_wmo,int64
surface_pressure_hpa,float64
cloud_cover_pct,int64
visibility_m,float64


In [23]:
# ==========================================================
# Hourly Coverage and Continuity Validation
# ==========================================================

weather_clean = weather_clean.sort_values('time').reset_index(drop=True)

expected_start = pd.Timestamp('2024-09-01 00:00')
expected_end = pd.Timestamp('2025-02-28 23:00')
expected_index = pd.date_range(expected_start, expected_end, freq='h')
observed_index = pd.DatetimeIndex(weather_clean['time'])
missing_hours = expected_index.difference(observed_index)
unexpected_hours = observed_index.difference(expected_index)

coverage_summary = pd.Series({
    'expected_start': expected_start,
    'observed_start': observed_index.min(),
    'expected_end': expected_end,
    'observed_end': observed_index.max(),
    'expected_hourly_records': len(expected_index),
    'observed_records': len(observed_index),
    'missing_hours': len(missing_hours),
    'unexpected_hours': len(unexpected_hours)
})
display(coverage_summary.to_frame(name='value'))

if len(missing_hours) or len(unexpected_hours):
    raise ValueError('Hourly coverage differs from the expected research period.')

print('Hourly continuity validated for the full research period.')

,value
expected_start,2024-09-01 00:00:00
observed_start,2024-09-01 00:00:00
expected_end,2025-02-28 23:00:00
observed_end,2025-02-28 23:00:00
expected_hourly_records,4344
observed_records,4344
missing_hours,0
unexpected_hours,0


Hourly continuity validated for the full research period.


## Export Cleaned Weather Dataset

After passing the validation checks, the standardised weather dataset is exported to the processed-data directory. This creates a stable, documented input for the subsequent metro-weather merge while preserving the raw Open-Meteo download unchanged.

In [24]:
# ==========================================================
# Export Validated Weather Dataset
# ==========================================================

processed_data_dir = project_root / 'DataProcessed'
processed_data_dir.mkdir(parents=True, exist_ok=True)
weather_output_path = processed_data_dir / 'weather_hourly_cleaned.csv'

weather_clean.to_csv(weather_output_path, index=False)
print(f'Cleaned weather dataset exported to: {weather_output_path}')
print(f'Exported shape: {weather_clean.shape}')

Cleaned weather dataset exported to: /Users/shivaya/Desktop/Research/DelhiMetroResearch/DataProcessed/weather_hourly_cleaned.csv
Exported shape: (4344, 11)
